In [1]:
import pandas as pd

#


In [2]:
import pandas as pd
import numpy as np

# Load the data
df = pd.read_csv(r"D:\Ahmed\My Projects\telecom intelligence project\data\processed\customer_features.csv")

print(f"Original data shape: {df.shape}")
print(f"\nNumerical columns:")
numerical_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print(numerical_cols)

Original data shape: (10000, 38)

Numerical columns:
['age', 'monthly_charges', 'age_missing', 'tenure_months', 'signed_up_after_window_start', 'call_minutes_mean', 'call_minutes_std', 'call_minutes_min', 'call_minutes_max', 'call_minutes_slope', 'call_minutes_recent_vs_early', 'call_minutes_cv', 'data_usage_gb_mean', 'data_usage_gb_std', 'data_usage_gb_min', 'data_usage_gb_max', 'data_usage_gb_slope', 'data_usage_gb_recent_vs_early', 'data_usage_gb_cv', 'sms_count_mean', 'sms_count_std', 'sms_count_min', 'sms_count_max', 'sms_count_slope', 'sms_count_recent_vs_early', 'sms_count_cv', 'complaints_total', 'complaints_mean', 'complaint_months', 'complaints_recent3', 'active_months', 'data_gb_per_charge', 'minutes_per_charge', 'churn']


In [3]:
# Function to detect outliers using IQR method
def detect_outliers_iqr(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = df[(df[column] < lower_bound) | (df[column] > upper_bound)]
    return outliers, lower_bound, upper_bound

# Detect outliers for each numerical column
print("Outlier Detection Summary:")
print("=" * 80)

outlier_summary = {}
for col in numerical_cols:
    outliers, lower, upper = detect_outliers_iqr(df, col)
    outlier_count = len(outliers)
    outlier_pct = (outlier_count / len(df)) * 100
    outlier_summary[col] = {
        'count': outlier_count,
        'percentage': outlier_pct,
        'lower_bound': lower,
        'upper_bound': upper
    }
    print(f"\n{col}:")
    print(f"  Lower bound: {lower:.4f}")
    print(f"  Upper bound: {upper:.4f}")
    print(f"  Outliers: {outlier_count} ({outlier_pct:.2f}%)")

Outlier Detection Summary:

age:
  Lower bound: -6.5000
  Upper bound: 93.5000
  Outliers: 0 (0.00%)

monthly_charges:
  Lower bound: 8.7437
  Upper bound: 91.5538
  Outliers: 111 (1.11%)

age_missing:
  Lower bound: -1.5000
  Upper bound: 2.5000
  Outliers: 0 (0.00%)

tenure_months:
  Lower bound: -29.0000
  Upper bound: 107.0000
  Outliers: 0 (0.00%)

signed_up_after_window_start:
  Lower bound: 0.0000
  Upper bound: 0.0000
  Outliers: 751 (7.51%)

call_minutes_mean:
  Lower bound: 224.5260
  Upper bound: 772.6049
  Outliers: 57 (0.57%)

call_minutes_std:
  Lower bound: 14.9121
  Upper bound: 81.2535
  Outliers: 77 (0.77%)

call_minutes_min:
  Lower bound: 142.6738
  Upper bound: 707.4638
  Outliers: 55 (0.55%)

call_minutes_max:
  Lower bound: 290.2587
  Upper bound: 854.3087
  Outliers: 53 (0.53%)

call_minutes_slope:
  Lower bound: -17.3525
  Upper bound: 17.4769
  Outliers: 68 (0.68%)

call_minutes_recent_vs_early:
  Lower bound: 0.7825
  Upper bound: 1.2219
  Outliers: 186 (1.86

In [4]:
# Remove outliers from numerical columns
print("\nRemoving outliers...")
print("=" * 80)

df_clean = df.copy()
outliers_removed_total = 0

for col in numerical_cols:
    outliers, lower, upper = detect_outliers_iqr(df_clean, col)
    outlier_count = len(outliers)
    
    if outlier_count > 0:
        # Cap outliers at the bounds instead of removing rows
        # This preserves the data while handling extreme values
        df_clean[col] = df_clean[col].clip(lower=lower, upper=upper)
        outliers_removed_total += outlier_count
        print(f"{col}: Capped {outlier_count} outliers to bounds [{lower:.4f}, {upper:.4f}]")

print(f"\nTotal outliers capped: {outliers_removed_total}")
print(f"Final data shape: {df_clean.shape}")


Removing outliers...
monthly_charges: Capped 111 outliers to bounds [8.7437, 91.5538]
signed_up_after_window_start: Capped 751 outliers to bounds [0.0000, 0.0000]
call_minutes_mean: Capped 57 outliers to bounds [224.5260, 772.6049]
call_minutes_std: Capped 77 outliers to bounds [14.9121, 81.2535]
call_minutes_min: Capped 55 outliers to bounds [142.6738, 707.4638]
call_minutes_max: Capped 53 outliers to bounds [290.2587, 854.3087]
call_minutes_slope: Capped 68 outliers to bounds [-17.3525, 17.4769]
call_minutes_recent_vs_early: Capped 186 outliers to bounds [0.7825, 1.2219]
call_minutes_cv: Capped 263 outliers to bounds [0.0120, 0.1856]
data_usage_gb_mean: Capped 57 outliers to bounds [7.3011, 12.6967]
data_usage_gb_std: Capped 60 outliers to bounds [0.9037, 4.9078]
data_usage_gb_min: Capped 110 outliers to bounds [0.7350, 10.4550]
data_usage_gb_max: Capped 97 outliers to bounds [9.7000, 19.1400]
data_usage_gb_slope: Capped 63 outliers to bounds [-1.0430, 1.0428]
data_usage_gb_recent_v

In [5]:
# Verify the cleaned data
print("\nVerification - Statistics after outlier removal:")
print("=" * 80)

print("\nBefore vs After comparison for key columns:")
key_cols = ['monthly_charges', 'call_minutes_mean', 'data_usage_gb_mean', 'sms_count_mean']
for col in key_cols:
    if col in df.columns:
        print(f"\n{col}:")
        print(f"  Before - Min: {df[col].min():.4f}, Max: {df[col].max():.4f}, Mean: {df[col].mean():.4f}")
        print(f"  After  - Min: {df_clean[col].min():.4f}, Max: {df_clean[col].max():.4f}, Mean: {df_clean[col].mean():.4f}")

print("\n\nSample of cleaned data:")
print(df_clean.head())


Verification - Statistics after outlier removal:

Before vs After comparison for key columns:

monthly_charges:
  Before - Min: -6.7200, Max: 390.3000, Mean: 51.0002
  After  - Min: 8.7437, Max: 91.5538, Mean: 50.1751

call_minutes_mean:
  Before - Min: 114.1244, Max: 865.0489, Mean: 498.6531
  After  - Min: 224.5260, Max: 772.6049, Mean: 498.6786

data_usage_gb_mean:
  Before - Min: 6.3322, Max: 14.8178, Mean: 10.0025
  After  - Min: 7.3011, Max: 12.6967, Mean: 10.0024

sms_count_mean:
  Before - Min: 6.2222, Max: 40.3333, Mean: 24.5304
  After  - Min: 11.2222, Max: 37.8889, Mean: 24.5308


Sample of cleaned data:
  customer_id   age  gender    region contract_type  monthly_charges  \
0      CUST_0   NaN    Male     Urban        Yearly            48.18   
1      CUST_1   NaN  Female  Suburban       Prepaid            38.94   
2     CUST_10   NaN  Female     Urban       Prepaid            47.81   
3    CUST_100  62.0  Female     Urban       Monthly            29.80   
4   CUST_1000   

In [6]:
# # Save the cleaned data
# output_path = r"D:\Ahmed\My Projects\telecom intelligence project\data\processed\customer_features_clean.csv"
# df_clean.to_csv(output_path, index=False)

# print(f"\nCleaned data saved to: {output_path}")
# print(f"Final shape: {df_clean.shape}")
# print(f"Columns: {df_clean.columns.tolist()}")